In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm.auto import tqdm
import json
def seed_everything(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


torch.backends.cudnn.benchmark = True
seed_everything(123)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")
amp_device = "cuda" if use_amp else "cpu"

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# Config
# -------------------------
WEIGHTS_ROOT = "./out"   # rep1..rep10/*
OUT_DIR = "./dev_ig_by_rep"
WEIGHTS_ROOT2 = "./data"
os.makedirs(OUT_DIR, exist_ok=True)




with open("../script_description/threshold_manifest_03.json", "r") as f:
    data = json.load(f)
REP_THRESH = {int(k): v for k, v in data["thresholds"].items()}


SEQ_LEN = 256
N_FILL = 0.25

BATCH_SIZE = 512
NUM_WORKERS = 0
PIN_MEMORY = (device.type == "cuda")

IG_BATCH = 64
TOP_PCT_BY_FINAL = 1.0  
IG_STEPS = 50
WINDOW = 10
TOPK_WINDOWS = 3
EDGE_TRIM = 5
BASELINE_VALUE = 0.25


_lut = np.full(256, -1, dtype=np.int16)
for ch, idx in [("A", 0), ("C", 1), ("G", 2), ("T", 3)]:
    _lut[ord(ch)] = idx
    _lut[ord(ch.lower())] = idx

def one_hot_encode_fast_with_len(seq: str, L: int = 256, n_fill: float = 0.25):
    if not isinstance(seq, str):
        seq = str(seq)
    true_len = min(len(seq), L)
    if len(seq) >= L:
        s = seq[:L]
    else:
        s = seq + ("N" * (L - len(seq)))
    b = np.frombuffer(s.encode("ascii", "replace"), dtype=np.uint8)
    if b.size < L:
        b = np.pad(b, (0, L - b.size), constant_values=ord("N"))
    elif b.size > L:
        b = b[:L]
    idx = _lut[b]
    x = np.full((4, L), n_fill, dtype=np.float32)
    pos = np.where(idx >= 0)[0]
    if pos.size > 0:
        x[:, pos] = 0.0
        x[idx[pos], pos] = 1.0
    return x, int(true_len)


class DNACNN(nn.Module):
    def __init__(self, in_ch=4, dropout=0.3, padding_mode="replicate"):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=7, padding=3, padding_mode=padding_mode),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=7, padding=3, padding_mode=padding_mode),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=7, padding=6, dilation=2, padding_mode=padding_mode),
            nn.BatchNorm1d(256),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
        )

    def masked_pool(self, x, lens):
        B, C, Lp = x.shape
        lens_p = (lens + 3) // 4  # ceil(lens/4)
        lens_p = torch.clamp(lens_p, min=1, max=Lp)
        t = torch.arange(Lp, device=x.device).view(1, 1, Lp)
        m = (t < lens_p.view(B, 1, 1))
        x_max = x.masked_fill(~m, -1e4).amax(dim=2, keepdim=True)
        x_sum = (x * m).sum(dim=2, keepdim=True)
        denom = lens_p.view(B, 1, 1).to(x.dtype)
        x_avg = x_sum / denom
        return x_max, x_avg

    def forward(self, x, lens):
        x = self.features(x)
        mx, av = self.masked_pool(x, lens)
        x = torch.cat([mx, av], dim=1)
        x = self.head(x)
        return x.squeeze(1)

def load_model(weights_path: str) -> nn.Module:
    model = DNACNN(dropout=0.30, padding_mode="replicate").to(device)
    sd = torch.load(weights_path, map_location=device)
    if isinstance(sd, dict) and "model_state" in sd:
        sd = sd["model_state"]
    model.load_state_dict(sd, strict=True)
    model.eval()
    return model


class DevDatasetWithSource(Dataset):
    def __init__(self, X_np: np.ndarray, lens_np: np.ndarray, seqs: list, source: list):
        self.X = X_np
        self.lens = lens_np
        self.seqs = seqs
        self.source = source

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]),
            torch.as_tensor(self.lens[idx], dtype=torch.long),
            self.seqs[idx],
            self.source[idx],
        )


_WINDOW_KERNEL_CACHE = {}

def sliding_window_sum_1d(score_1d: torch.Tensor, window: int) -> torch.Tensor:
    L = score_1d.numel()
    if L < window:
        return torch.empty(0, device=score_1d.device, dtype=score_1d.dtype)
    key = (window, score_1d.device, score_1d.dtype)
    w = _WINDOW_KERNEL_CACHE.get(key)
    if w is None:
        w = torch.ones((1, 1, window), device=score_1d.device, dtype=score_1d.dtype)
        _WINDOW_KERNEL_CACHE[key] = w
    x = score_1d.view(1, 1, L)
    y = torch.nn.functional.conv1d(x, w, stride=1, padding=0)
    return y.view(-1)


def integrated_gradients_batch(model, x, lens, steps=50, baseline_value=0.25):
    if steps <= 1:
        raise ValueError("steps must be > 1")
    model.eval()
    x = x.detach().float()
    lens = lens.detach().long()
    baseline = torch.full_like(x, float(baseline_value), dtype=torch.float32)

    alphas = torch.linspace(0.0, 1.0, steps, device=x.device, dtype=torch.float32)
    weights = torch.ones_like(alphas)
    weights[0] = 0.5
    weights[-1] = 0.5
    wsum = weights.sum()

    total_grad = torch.zeros_like(x, dtype=torch.float32)
    for a, w in zip(alphas, weights):
        x_interp = baseline + a * (x - baseline)
        x_interp.requires_grad_(True)
        logits = model(x_interp, lens)
        grads = torch.autograd.grad(outputs=logits.sum(), inputs=x_interp)[0]
        total_grad += w * grads.detach()

    avg_grad = total_grad / wsum
    return (x - baseline) * avg_grad

@torch.no_grad()
def importance_from_attr_batch(attr, lens, window=10, topk=3, edge_trim=5):
    B, _, _ = attr.shape
    score = attr.abs().sum(dim=1)  # (B,L)
    importance = np.zeros((B,), dtype=np.float32)
    best_pos = np.zeros((B,), dtype=np.int64)

    for i in range(B):
        Li = int(lens[i].item())
        s = score[i, :Li].clone()
        if edge_trim > 0 and s.numel() > 2 * edge_trim:
            s[:edge_trim] = 0
            s[-edge_trim:] = 0
        wscore = sliding_window_sum_1d(s, window=window)
        if wscore.numel() == 0:
            importance[i] = float(s.sum().item())
            best_pos[i] = 0
            continue
        k = min(int(topk), int(wscore.numel()))
        topk_vals, topk_idx = torch.topk(wscore, k=k)
        importance[i] = float(topk_vals.mean().item())
        best_pos[i] = int(topk_idx[0].item())
    return importance, best_pos

def screen_dev_by_final_score_microbatch(
    model, loader, prob_thresh, top_pct_by_final,
    window, topk_windows, ig_steps, edge_trim,
    baseline_value, ig_batch
):
    if not (0.0 < float(top_pct_by_final) <= 1.0):
        raise ValueError("top_pct_by_final must be in (0,1].")

    dev = next(model.parameters()).device
    model.eval()

    cand = []
    total = 0

    pbar1 = tqdm(loader, desc=f"Stage 1 (prob thr={prob_thresh:.2f})", unit="batch")
    for xb, lb, seqb, srcb in pbar1:
        total += len(seqb)
        xb = xb.to(dev, non_blocking=True)
        lb = lb.to(dev, non_blocking=True).view(-1)

        with torch.no_grad():
            with autocast(amp_device, enabled=use_amp):
                probs = torch.sigmoid(model(xb, lb))

        probs_cpu = probs.detach().float().cpu().numpy()
        idxs = np.where(probs_cpu > float(prob_thresh))[0]
        if idxs.size == 0:
            pbar1.set_postfix(total=total, cand=len(cand))
            continue

        xb_cpu = xb.detach().cpu()
        lb_cpu = lb.detach().cpu()
        for i in idxs.tolist():
            cand.append({
                "sequence": seqb[i],
                "source": srcb[i],
                "prob": float(probs_cpu[i]),
                "x_cpu": xb_cpu[i],
                "l_cpu": lb_cpu[i],
            })
        pbar1.set_postfix(total=total, cand=len(cand))

    print(f"DEV total: {total} | candidates prob>{prob_thresh:.2f}: {len(cand)}")
    if not cand:
        return pd.DataFrame(columns=[
            "sequence","source","prob","importance","final_score",
            "best_window_start","best_window_end","best_window_seq"
        ])

    if dev.type == "cuda":
        t = torch.randn(1, device=dev, requires_grad=True)
        (t * t).sum().backward()
        torch.cuda.synchronize()

    records = []
    n = len(cand)
    pbar2 = tqdm(range(0, n, ig_batch), desc="Stage 2 (IG)", unit="chunk")

    for start in pbar2:
        chunk = cand[start:start + ig_batch]
        x_batch = torch.stack([c["x_cpu"] for c in chunk], dim=0).to(dev, non_blocking=True)
        l_batch = torch.stack([c["l_cpu"] for c in chunk], dim=0).to(dev, non_blocking=True).view(-1).long()

        ctx = torch.cuda.amp.autocast(enabled=False) if dev.type == "cuda" else torch.amp.autocast("cpu", enabled=False)
        with ctx:
            attr = integrated_gradients_batch(
                model=model, x=x_batch, lens=l_batch,
                steps=ig_steps, baseline_value=baseline_value
            )

        imp_np, best_pos_np = importance_from_attr_batch(
            attr=attr, lens=l_batch,
            window=window, topk=topk_windows, edge_trim=edge_trim
        )

        for i, c in enumerate(chunk):
            seq = c["sequence"]
            best_pos = int(best_pos_np[i])
            best_end = best_pos + int(window)
            prob = float(c["prob"])
            importance = float(imp_np[i])
            records.append({
                "sequence": seq,
                "source": c["source"],
                "prob": prob,
                "importance": importance,
                "final_score": prob * importance,
                "best_window_start": best_pos,
                "best_window_end": best_end,
                "best_window_seq": seq[best_pos:best_end] if len(seq) >= best_end else "",
            })

        pbar2.set_postfix(done=min(start + ig_batch, n), total=n)

    out = pd.DataFrame(records).sort_values("final_score", ascending=False).reset_index(drop=True)
    k = max(1, int(np.ceil(len(out) * float(top_pct_by_final))))
    return out.head(k).reset_index(drop=True)

# -------------------------
# -------------------------
def load_rep_dev_sequences(train_csv: str, val_csv: str):
    df_tr = pd.read_csv(train_csv)
    df_va = pd.read_csv(val_csv)
    if "sequence" not in df_tr.columns or "sequence" not in df_va.columns:
        raise ValueError(f"Missing 'sequence' column in {train_csv} or {val_csv}")

    seqs_tr = df_tr["sequence"].astype(str).tolist()
    seqs_va = df_va["sequence"].astype(str).tolist()
    seqs = seqs_tr + seqs_va
    srcs = (["train_dev"] * len(seqs_tr)) + (["val_dev"] * len(seqs_va))
    return seqs, srcs

def encode_sequences(sequences: list[str], seq_len: int, n_fill: float):
    tmp = [one_hot_encode_fast_with_len(s, seq_len, n_fill=n_fill) for s in tqdm(sequences, desc="Encoding", unit="seq")]
    X = np.stack([t[0] for t in tmp]).astype(np.float32)
    lens = np.array([t[1] for t in tmp], dtype=np.int64)
    return X, lens

# ============================================================
# ============================================================
if __name__ == "__main__":
    for rep in range(1, 11):
        rep_dir = os.path.join(WEIGHTS_ROOT, f"rep{rep}")
        rep_dir2 = os.path.join(WEIGHTS_ROOT2, f"rep{rep}")
        weights_path = os.path.join(rep_dir, "best_model.pt")
        train_csv = os.path.join(rep_dir2, "train.csv")
        val_csv = os.path.join(rep_dir2, "val.csv")

        if not os.path.exists(weights_path):
            print(f"[SKIP] rep{rep} missing weights: {weights_path}")
            continue
        if not (os.path.exists(train_csv) and os.path.exists(val_csv)):
            print(f"[SKIP] rep{rep} missing dev csv: {train_csv} or {val_csv}")
            continue

        thr = float(REP_THRESH[rep])
        print(f"\n==================== REP {rep} DEV IG (thr={thr:.2f}) ====================")

        # load dev sequences for this rep
        sequences, sources = load_rep_dev_sequences(train_csv, val_csv)

        # encode (rep-specific, because dev is different per rep)
        X, lens = encode_sequences(sequences, seq_len=SEQ_LEN, n_fill=N_FILL)
        print("len min/med/max:", int(lens.min()), int(np.median(lens)), int(lens.max()))

        dataset = DevDatasetWithSource(X, lens, sequences, sources)
        loader = DataLoader(
            dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
        )

        model = load_model(weights_path)

        top_df = screen_dev_by_final_score_microbatch(
            model=model,
            loader=loader,
            prob_thresh=thr,  
            top_pct_by_final=TOP_PCT_BY_FINAL,
            window=WINDOW,
            topk_windows=TOPK_WINDOWS,
            ig_steps=IG_STEPS,
            edge_trim=EDGE_TRIM,
            baseline_value=BASELINE_VALUE,
            ig_batch=IG_BATCH,
        )

        out_csv = os.path.join(OUT_DIR, f"rep{rep}_dev_top_by_final.csv")
        top_df.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}  rows={len(top_df)}")

    print("\nAll done. Outputs in:", os.path.abspath(OUT_DIR))


Device: cuda
GPU: NVIDIA GeForce RTX 5090

==================== REP 1 DEV IG (thr=0.44) ====================


Encoding: 100%|██████████| 17684/17684 [00:00<00:00, 93180.29seq/s]


len min/med/max: 100 100 100


Stage 1 (prob thr=0.44): 100%|██████████| 35/35 [00:02<00:00, 12.15batch/s, cand=9323, total=17684]


DEV total: 17684 | candidates prob>0.44: 9323


...

len min/med/max: 100 100 100


Stage 1 (prob thr=0.50): 100%|██████████| 35/35 [00:01<00:00, 24.89batch/s, cand=9437, total=17680] 


DEV total: 17680 | candidates prob>0.50: 9437


Stage 2 (IG):   0%|          | 0/148 [00:00<?, ?chunk/s]/tmp/ipykernel_556587/3843277112.py:302: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx = torch.cuda.amp.autocast(enabled=False) if dev.type == "cuda" else torch.amp.autocast("cpu", enabled=False)
Stage 2 (IG): 100%|██████████| 148/148 [00:31<00:00,  4.70chunk/s, done=9437, total=9437]

Saved: ./dev_ig_by_rep/rep10_dev_top_by_final.csv  rows=9437

All done. Outputs in: /root/autodl-tmp/10_rep_final2/dev_ig_by_rep


In [ ]:
import os
import pandas as pd
from collections import Counter
import numpy as np

IN_DIR = "./dev_ig_by_rep"  
FILE_TMPL = "rep{rep}_dev_top_by_final.csv" 
SEQ_COL = "sequence"
SCORE_COL = "final_score"

K = 5
TOP_PCT = 0.8  
REPS = range(1, 11)

OUT_COMBINED = f"combined_{K}mer_analysis_top{int(TOP_PCT*100)}pct.csv"
OUT_PER_REP_SUMMARY = f"per_rep_{K}mer_summary_top{int(TOP_PCT*100)}pct.csv"

valid_chars = set("ACGT")


def load_top_seqs(path: str, top_pct: float) -> list[str]:
    df = pd.read_csv(path)
    if SCORE_COL not in df.columns:
        raise ValueError(f"{path}: '{SCORE_COL}' column not found")
    if SEQ_COL not in df.columns:
        raise ValueError(f"{path}: '{SEQ_COL}' column not found")

    df = df.sort_values(SCORE_COL, ascending=False).reset_index(drop=True)
    n_keep = max(1, int(np.ceil(len(df) * top_pct)))
    df = df.head(n_keep)
    seqs = df[SEQ_COL].astype(str).str.upper().tolist()
    return seqs, len(df)

def count_kmers(seqs: list[str], k: int):
    kmer_total = Counter()
    kmer_hit_sequences = Counter()
    total_kmer_instances = 0

    for seq in seqs:
        seen = set()
        L = len(seq)
        for i in range(L - k + 1):
            kmer = seq[i:i+k]
            if set(kmer) <= valid_chars:
                kmer_total[kmer] += 1
                total_kmer_instances += 1
                seen.add(kmer)
        for km in seen:
            kmer_hit_sequences[km] += 1

    return kmer_total, kmer_hit_sequences, total_kmer_instances


per_rep_frames = []
rep_meta_rows = []

for rep in REPS:
    fpath = os.path.join(IN_DIR, FILE_TMPL.format(rep=rep))
    if not os.path.exists(fpath):
        print(f"[SKIP] missing: {fpath}")
        continue

    seqs, n_used = load_top_seqs(fpath, TOP_PCT)
    kmer_total, kmer_hit, total_instances = count_kmers(seqs, K)

    rep_meta_rows.append({
        "rep": rep,
        "file": fpath,
        "n_sequences_used": n_used,
        "total_kmer_instances": total_instances
    })

    rows = []
    for km, cnt in kmer_total.items():
        rows.append({
            "rep": rep,
            "kmer": km,
            "total_occurrences": int(cnt),
            "hit_sequences": int(kmer_hit[km]),
            "frequency": float(cnt / max(total_instances, 1)),
        })
    per_rep_frames.append(pd.DataFrame(rows))

if not per_rep_frames:
    raise RuntimeError("No rep files found. Check IN_DIR/FILE_TMPL.")

per_rep_df = pd.concat(per_rep_frames, ignore_index=True)
rep_meta_df = pd.DataFrame(rep_meta_rows)


per_rep_df.to_csv(OUT_PER_REP_SUMMARY, index=False)
print("Saved per-rep long table:", OUT_PER_REP_SUMMARY)


agg_sum = per_rep_df.groupby("kmer", as_index=False).agg(
    total_occurrences_sum=("total_occurrences", "sum"),
    hit_sequences_sum=("hit_sequences", "sum"),
    reps_present=("rep", "nunique"),
)


total_instances_all = int(rep_meta_df["total_kmer_instances"].sum())
agg_sum["frequency_global"] = agg_sum["total_occurrences_sum"] / max(total_instances_all, 1)


freq_stats = per_rep_df.groupby("kmer", as_index=False).agg(
    freq_mean=("frequency", "mean"),
    freq_std=("frequency", "std"),
)
freq_stats["freq_std"] = freq_stats["freq_std"].fillna(0.0)
freq_stats["freq_cv"] = freq_stats["freq_std"] / (freq_stats["freq_mean"] + 1e-12)


n_rep_effective = int(rep_meta_df["rep"].nunique())
agg_sum["rep_coverage"] = agg_sum["reps_present"] / max(n_rep_effective, 1)


final = agg_sum.merge(freq_stats, on="kmer", how="left")


per_rep_df["rank_in_rep"] = per_rep_df.groupby("rep")["total_occurrences"]\
                                      .rank(method="min", ascending=False)
rank_stats = per_rep_df.groupby("kmer", as_index=False).agg(
    rank_mean=("rank_in_rep", "mean"),
    rank_median=("rank_in_rep", "median"),
)
final = final.merge(rank_stats, on="kmer", how="left")


final["score_combo"] = (
    (final["frequency_global"] * 1.0) *
    (final["rep_coverage"] ** 1.0) /
    (1.0 + final["freq_cv"])
)

final = final.sort_values(
    ["score_combo", "rep_coverage", "frequency_global", "total_occurrences_sum"],
    ascending=False
).reset_index(drop=True)

final.to_csv(OUT_COMBINED, index=False)
print("Saved combined analysis:", OUT_COMBINED)


print("\nEffective reps found:", n_rep_effective)
print("Total kmer instances across reps:", total_instances_all)

print("\nTop 20 by combo score:")
print(final.head(20)[[
    "kmer", "score_combo", "rep_coverage", "reps_present",
    "frequency_global", "freq_mean", "freq_std", "freq_cv",
    "total_occurrences_sum", "hit_sequences_sum",
    "rank_mean", "rank_median"
]])

print("\nTop 20 most stable (lowest CV, require appearing >= half reps):")
stable = final[final["reps_present"] >= max(1, n_rep_effective//2)] \
    .sort_values(["freq_cv", "rep_coverage", "frequency_global"], ascending=[True, False, False]) \
    .head(20)
print(stable[[
    "kmer", "freq_cv", "rep_coverage", "frequency_global", "total_occurrences_sum"
]])

Saved per-rep long table: per_rep_5mer_summary_top80pct.csv
Saved combined analysis: combined_5mer_analysis_top80pct.csv

Effective reps found: 10
Total kmer instances across reps: 7026336

Top 20 by combo score:
     kmer  score_combo  rep_coverage  reps_present  frequency_global  \
0   CTCTG     0.003582           1.0            10          0.003635   
1   CTGTG     0.003561           1.0            10          0.003633   
2   CACAG     0.003551           1.0            10          0.003596   
3   CAGAG     0.003350           1.0            10          0.003385   
4   CTGGG     0.003339           1.0            10          0.003378   
5   CAGCC     0.003309           1.0            10          0.003354   
6   GGCTG     0.003222           1.0            10          0.003276   
7   CCCAG     0.003218           1.0            10          0.003267   
8   CCCTG     0.003081           1.0            10          0.003143   
9   CAGGG     0.003038           1.0            10          0.00306